In [1]:
import re
import os
import numpy as np
import pandas as pd

import sys
sys.path.append("../../tools")
import prepare as prep
import functions as func


In [2]:
import importlib
#importlib.reload(func)

In [3]:
def single_processing(df, NumSpecies, MaxNum, a, sur_frac=False):
    df = func.clean_csv_missing(df, drop=True, verbose=True)
    df = func.DealWithOutlier_ver194(NumSpecies, a, df)
    df = func.pad_Xnum_and_ynum(df, NumSpecies, MaxNum, sur_frac)
    return df

def MergeSubFolders(rootdir, outdir, MaxNum=20, a=0.7, sur_frac=False):

    # Merge all CSV files in subfolders of rootdir into one CSV file in outdir
    os.makedirs(outdir, exist_ok=True)

    subfolders = []

    for f in os.listdir(rootdir):
        folderpath = os.path.join(rootdir, f)

        if not os.path.isdir(folderpath):
            continue

        # Extract richness from folder names such as "3" or "sp3"
        match = re.fullmatch(r"(?:sp)?(\d+)", f)

        if match:
            subfolders.append((int(match.group(1)), f))
        else:
            print(f"Skip {f}")

    # Sort folders by richness
    subfolders.sort(key=lambda x: x[0])

    for NumSpecies, folder in subfolders:

        folderpath = os.path.join(rootdir, folder)

        csvfiles = sorted(
            [
                f for f in os.listdir(folderpath)
                if f.endswith(".csv")
                and not f.startswith("merged_")
            ]
        )

        # Clean missing values, handle outliers, and pad Xnum and ynum
        dflist = []

        for file in csvfiles:

            filepath = os.path.join(folderpath, file)

            try:
                df = pd.read_csv(filepath)
                n0 = len(df)
                df = single_processing(df, NumSpecies, MaxNum, a, sur_frac)

                print(f"{folder}/{file}: {n0} -> {len(df)}")
                dflist.append(df)

            except Exception as e:
                print(f"Error: {filepath}")
                print(e)

        if len(dflist) == 0:
            print(f"No valid csv in {folder}")
            continue

        df_merge = pd.concat(dflist, axis=0, ignore_index=True)
        datasize = len(df_merge)

        filepath2 = os.path.join(
            outdir, f"{NumSpecies}-{MaxNum}_{datasize}.csv"
        )

        df_merge.to_csv(filepath2, index=False)

        print(f"Saved: {filepath2}")
        print(f"Shape = {df_merge.shape}")
        

In [ ]:
rootdir = r"../../RawData/results"
outdir = r"../../Databases/Ver1.9/ver1.9.6_re250ti800"
MergeSubFolders(rootdir, outdir, MaxNum=20, a=0.7, sur_frac=True)


In [ ]:
df = pd.read_csv("../../RawData/ver1.9.6/20_102400.csv")
result = single_processing(df, NumSpecies=20, MaxNum=20, a=0.7)
save_path = "../../Databases/Ver1.9/ver1.9.6_revision/20-20_102400.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
result.to_csv(save_path, index=False)


In [ ]:
for spe in range(3, 21):
    df = pd.read_csv(f"../../RawData/ver1.9.6/{spe}_102400.csv")
    result = single_processing(df, spe, 20, a=0.7)
    result.to_csv(f"../../Databases/Ver1.9/ver1.9.6.1_upto20/{spe}-20_102400.csv", index=False)


(102213, 25)
(101954, 35)
(101786, 47)
(101551, 61)
(101380, 77)
(101264, 95)
(101123, 115)
(101066, 137)
(100903, 161)
(100871, 187)
(100783, 215)
(100691, 245)
Row 17200, Column '242' has a missing value
Row 21600, Column '242' has a missing value
(100591, 277)
(100592, 311)
(100574, 347)
(100486, 385)
(100507, 425)
(100508, 467)
